# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 Mass Spectrometry EV Proteins](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema available at the following URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print(f"Source URL: {croissant_url}")

## 2. Data Overview
Review available record sets, field names, and their corresponding `@id` identifiers.
All entities are always referenced by their `@id` field for reproducibility and clarity.

In [ ]:
# List all record sets with their @id and fields
record_sets = list(dataset.metadata.recordSets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields (@id and name):")
    for field in rs.fields:
        print(f"    - {field.id}: {field.name} ({field.dataType})")
    print("\n---\n")

## 3. Data Extraction
Load the records from a specific record set using its `@id`, and create pandas DataFrames for further analysis.

**Example below loads all record sets and shows the first record set's columns. You can adjust the used record set by its `@id`.**

In [ ]:
# Collect all record set @ids for loading
record_set_ids = [rs.id for rs in dataset.metadata.recordSets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show the first record set and its columns as an example
example_record_set_id = record_set_ids[0]  # or change this to a different record set @id as needed
print(f"Columns in record set '{example_record_set_id}':")
print(dataframes[example_record_set_id].columns.tolist())
dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing values, and grouping if appropriate.

In this example, we select a numeric field (e.g., `cr:coverage_percent`) and a group field (e.g., `cr:sample_id`). Replace these with the actual `@id` values as needed for your analysis.

In [ ]:
# Select the record set and fields for analysis
# Inspect fields in section 2 above for the correct @ids
record_set_id = example_record_set_id
# Example field IDs (ensure these exist or update accordingly)
numeric_field_id = 'cr:coverage_percent'  # Replace with a real numeric field @id from your dataset
group_field_id = 'cr:sample_id'  # Replace with a real categorical field @id from your dataset

# If the field is missing, update the field @id accordingly
df = dataframes.get(record_set_id)
if df is not None and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Example group-by if possible
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in record set '{record_set_id}'. Please update with a valid numeric field @id from section 2 above.")

## 5. Visualization
Visualize the distribution of numeric fields and group summaries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if the numeric field exists
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} grouped by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print(f"Cannot plot because {numeric_field_id} not in columns.")

## 6. Conclusion
This notebook demonstrated loading, exploration, and basic EDA of a dataset following the Croissant schema using the `mlcroissant` library.

Key findings and analyses will depend on the domain-specific interpretation of the field values. For deeper analysis, refer to the dataset's documentation or FAIR^2 portal.

**Reminder:** Use proper `@id` throughout your workflow for accuracy and reproducibility.